In [1]:
import numpy as np
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Load your existing data
print("⏳ Loading Data...")
X = np.load("../data/processed/X_images.npy")
y = np.load("../data/processed/y_labels.npy")

# --- TRICK: Convert Grayscale (1 Channel) to RGB (3 Channels) ---
# MobileNet needs 3 channels. We just stack the same image 3 times.
print(f"Original Shape: {X.shape}")
X_rgb = np.repeat(X, 3, axis=-1) 
print(f"New RGB Shape:  {X_rgb.shape}") # Should be (N, 128, 216, 3)

# Split
X_train, X_test, y_train, y_test = train_test_split(X_rgb, y, test_size=0.2, random_state=42, stratify=y)
print("✅ Data Ready for Transfer Learning")

f:\Research\Project\Final\infant-growth-monitoring-system\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


⏳ Loading Data...
Original Shape: (1705, 128, 216, 1)
New RGB Shape:  (1705, 128, 216, 3)
✅ Data Ready for Transfer Learning


In [2]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.optimizers import Adam
from sklearn.utils import class_weight

# 1. Calculate Class Weights (Crucial for Imbalance)
y_integers = np.argmax(y_train, axis=1)
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_integers),
    y=y_integers
)
class_weight_dict = dict(enumerate(class_weights))

# 2. Load the Pre-Trained "Base" Model
# include_top=False means we cut off the "Head" (the part that classifies dogs/cats)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 216, 3))

# Freeze the base! We don't want to break what Google already taught it.
base_model.trainable = False 

# 3. Add our Custom "Baby Cry" Head
inputs = Input(shape=(128, 216, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)  # Dropout to prevent memorization
outputs = Dense(3, activation='softmax')(x) # 3 Classes

model = Model(inputs, outputs)

# 4. Compile
# We use a slightly higher learning rate because the base is frozen
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# 5. Train
print("\n🚀 Training with Transfer Learning...")
history = model.fit(X_train, y_train, 
                    epochs=25, 
                    batch_size=32, 
                    validation_data=(X_test, y_test),
                    class_weight=class_weight_dict)

# 6. Evaluate
loss, acc = model.evaluate(X_test, y_test)
print(f"\n🏆 Final Accuracy: {acc*100:.2f}%")



C:\Users\User\AppData\Local\Temp\ipykernel_27940\1191778358.py:19: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 216, 3))


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 12s 1us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 216, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 4, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │         3,843 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,261,827 (8.63 MB)

 Trainable params: 3,843 (15.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)


🚀 Training with Transfer Learning...
Epoch 1/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 20s 297ms/step - accuracy: 0.3453 - loss: 1.4634 - val_accuracy: 0.3871 - val_loss: 1.1113
Epoch 2/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 12s 277ms/step - accuracy: 0.3886 - loss: 1.3167 - val_accuracy: 0.4135 - val_loss: 1.0847
Epoch 3/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 22s 304ms/step - accuracy: 0.4186 - loss: 1.2006 - val_accuracy: 0.3842 - val_loss: 1.1264
Epoch 4/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 14s 329ms/step - accuracy: 0.4399 - loss: 1.1693 - val_accuracy: 0.4076 - val_loss: 1.0950
Epoch 5/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 16s 384ms/step - accuracy: 0.4326 - loss: 1.1258 - val_accuracy: 0.4985 - val_loss: 1.0324
Epoch 6/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 21s 487ms/step - accuracy: 0.4641 - loss: 1.0993 - val_accuracy: 0.4839 - val_loss: 1.0392
Epoch 7/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 38s 412ms/step - accuracy: 0.4765 - loss: 1.0575 - val_accuracy: 0.4047 - val_loss: 1.0772
Epoch 8/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 22s 507ms/step - accu